# Stage 09 on Colab: the convergence control and Fig. S4

This is the **only** stage of the reproducibility module that wants a GPU. It retrains
the five graph architectures with the epoch ceiling raised from 30 to 1,000 and the
capacity restored to reference scale: fifteen runs, about **40 h on eight CPU cores**
against roughly **4 h on one T4**.

Everything else in the module runs on a laptop. Stage 03 is network-bound on the ChEMBL
API and stage 07 is exact TreeSHAP on a random forest, which has no GPU path at all.

Before you start: **Runtime -> Change runtime type -> GPU**.

## Step 0 - mount Drive first

Colab recycles runtimes. A four-hour run written to the local VM disk is lost entirely
when that happens; written to Drive it resumes from its last checkpoint. Do not skip
this cell, and do not start the run before it has succeeded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = '/content/drive/MyDrive/glioma'
os.environ['MARINE_REPRO_DATA'] = f'{ROOT}/data'          # the large inputs
os.environ['GLIOMA_CKPT']       = f'{ROOT}/checkpoints'   # resume state
os.environ['GLIOMA_CACHE']      = f'{ROOT}/cache'         # fingerprint cache
for p in ('data', 'checkpoints', 'cache', 'outputs'):
    pathlib.Path(f'{ROOT}/{p}').mkdir(parents=True, exist_ok=True)

print('data dir   :', os.environ['MARINE_REPRO_DATA'])
print('checkpoints:', os.environ['GLIOMA_CKPT'])
print('already checkpointed:', sorted(os.listdir(os.environ['GLIOMA_CKPT']))[:10])

## Step 1 - unpack the module and install

Upload `marine_antiglioma_reproducibility.tar.gz` to `MyDrive/glioma/` first, or copy it
in with the file browser.

In [ ]:
!cp /content/drive/MyDrive/glioma/marine_antiglioma_reproducibility.tar.gz /content/
!tar xzf /content/marine_antiglioma_reproducibility.tar.gz -C /content/
%cd /content/marine_antiglioma_reproducibility
!pip install -q -r env/requirements.txt
import subprocess
print(subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip() or 'NO GPU VISIBLE')

## Step 2 - confirm the inputs resolve before spending four hours

`--check` resolves every input and runs nothing. If it names a missing file, put that
file in `MyDrive/glioma/data/` and re-run this cell; `config/inputs_manifest.tsv` says
where each one comes from.

In [ ]:
!./run_stage.sh 09 --check

## Step 3 - the run

About 4 h on a T4. Safe to interrupt: the per-run checkpoints under `GLIOMA_CKPT` mean
re-running this cell resumes rather than restarting.

If you drive the shell with `!` rather than `%%bash`, note that each `!` line is its own
subshell, so `!export VAR=x` on a line by itself is forgotten the moment that line ends.
The variables set through `os.environ` in step 0 are inherited correctly.

In [ ]:
!./run_stage.sh 09

## Step 4 - Fig. S4, and verify nothing moved

Stage 11 writes a vector PDF and a 300 dpi PNG. Stage 12 re-asserts the published
headline numbers and prints PASS/FAIL per item - run it after any stage that touches
a reported value.

In [ ]:
!./run_stage.sh 11 --only S4
!./run_stage.sh 12

In [ ]:
import shutil, glob, os
dst = '/content/drive/MyDrive/glioma/outputs'
os.makedirs(dst, exist_ok=True)
for f in glob.glob('outputs/stage11_figures/figS4*') + glob.glob('outputs/stage12_verify/*.csv'):
    shutil.copy(f, dst)
print('copied to Drive:', sorted(os.path.basename(f) for f in os.listdir(dst)))

## If you also want stage 10 on GPU

Stage 10 is the hyperparameter-budget control: ~30 h CPU, ~5 h GPU, already checkpointed
per configuration. Only XGBoost trains on the GPU there - the reported random forest is
scikit-learn on CPU by design, because the interpretation chapter consumes a
scikit-learn forest. Run it the same way:

```
!./run_stage.sh 10
```

Neither stage 09 nor stage 10 changes any main-text result. They are controls that
answer "was the comparison compute-limited or under-tuned", and both answered no.